In [5]:
# Uses:
#   1. population-dataset.csv
#   2. MetroManila_Barangay_Flood_Area_Validated.csv
#   3. manila_final_base_dataset_NEW.csv

# Output:
#   merged-dataset.csv
#   merged-dataset.xlsx

import pandas as pd
import numpy as np
import re

In [6]:
# 0. Helper Functions

def clean_barangay_name(x):
    """
    Standardizes barangay names for merging.

    Examples:
    'Brgy. 105'       -> 'BARANGAY 105'
    'Barangay 202-A'  -> 'BARANGAY 202-A'
    'barangay 105 '   -> 'BARANGAY 105'
    """
    x = str(x).upper().strip()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("BRGY.", "BARANGAY")
    x = x.replace("BRGY", "BARANGAY")
    return x


def clean_barangay_name_strict(x):
    """
    Stricter matching key.
    Removes spaces and hyphens.

    Example:
    'Barangay 202-A' -> 'BARANGAY202A'
    'Barangay 202A'  -> 'BARANGAY202A'
    """
    x = clean_barangay_name(x)
    x = x.replace("-", "")
    x = x.replace(" ", "")
    return x


def extract_number(text):
    """
    Extracts numeric barangay number for sorting.
    Non-number entries such as Tutuban Mall return 0.
    """
    match = re.search(r"(\d+)", str(text))
    return int(match.group()) if match else 0


def safe_numeric(df, cols):
    """
    Converts selected columns to numeric if they exist.
    """
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


In [7]:
# 1. File Paths

population_path = "/content/population-dataset.csv"
flood_path = "/content/MetroManila_Barangay_Flood_Area_Validated.csv"
base_path = "/content/manila_final_base_dataset_NEW.csv"

output_csv = "/content/manila-city-flood-population_enriched_NEW.csv"
output_excel = "/content/manila-city-flood-population_enriched_NEW.xlsx"

In [8]:
# ============================================================
# 2. Load 2024 Population Dataset
# ============================================================

df_pop = pd.read_csv(population_path, header=None)

# Some rows may be blank separator rows in the CSV.
# Keep only first two columns.
df_pop = df_pop.iloc[:, :2]
df_pop.columns = ["Barangay", "Population 2024"]

# Clean barangay names
df_pop["Barangay"] = df_pop["Barangay"].astype(str).str.strip()

# Remove blank/separator rows
df_pop = df_pop[
    (df_pop["Barangay"].notna()) &
    (df_pop["Barangay"].str.strip() != "") &
    (df_pop["Barangay"].str.lower() != "nan")
].copy()

# Convert population to numeric
df_pop["Population 2024"] = pd.to_numeric(
    df_pop["Population 2024"],
    errors="coerce"
)

# Remove rows without population values
df_pop = df_pop[df_pop["Population 2024"].notna()].copy()

# Optional: remove non-barangay summary rows if needed
# Example: "Intramuros" may appear as a district/area summary, not a barangay.
# Keep it only if it exists in your flood dataset and base dataset.
# Otherwise, it will naturally be removed by inner merge with flood dataset.

print("2024 population dataset loaded.")
print("Population shape:", df_pop.shape)
print(df_pop.head())


2024 population dataset loaded.
Population shape: (898, 2)
     Barangay  Population 2024
0  Barangay 1           3119.0
1  Barangay 2           1666.0
2  Barangay 3           1232.0
3  Barangay 4           1717.0
4  Barangay 5           1321.0


In [9]:
# 3. Load Flood Dataset

df_flood = pd.read_csv(flood_path, header=None)

df_flood.columns = [
    "Barangay",
    "Barangay Area Sqm",
    "Flood Area 5yr Sqm",
    "Flood PCT 5yr",
    "Flood Area 25yr Sqm",
    "Flood PCT 25yr",
    "Flood Area 100yr Sqm",
    "Flood PCT 100yr",
    "Flag 25yr Less than 5yr",
    "Flag 100yr Less than 25yr",
    "Flag 100yr Less than 5yr",
    "Barangay No"
]

df_flood = df_flood.iloc[1:].copy()

df_flood["Barangay"] = df_flood["Barangay"].astype(str).str.strip()

print("\nFlood dataset loaded.")
print("Flood shape:", df_flood.shape)
print(df_flood.head())


Flood dataset loaded.
Flood shape: (897, 12)
       Barangay   Barangay Area Sqm Flood Area 5yr Sqm       Flood PCT 5yr  \
1    Barangay 1    49547.4905221462  8264.845529826996  16.680654141571235   
2   Barangay 10   6057.586255228475  5544.134717214669   91.52382621756922   
3  Barangay 100  29349.500167627426   8985.70692044154   30.61621788827872   
4  Barangay 101  103997.36954502873  37525.24293193707   36.08287699593152   
5  Barangay 102   23899.67583607758  4506.429582428609  18.855609646495548   

  Flood Area 25yr Sqm     Flood PCT 25yr Flood Area 100yr Sqm  \
1  21069.565046191274  42.52398017367565    24194.31705659861   
2   5999.129221978799  99.03497811196299     6033.96419192838   
3  14348.687691315523  48.88903596096728   15223.078875703362   
4   57378.99725640311  55.17350824105141    61606.83973598299   
5  14200.178502915202  59.41577869219225   20019.915620312564   

     Flood PCT 100yr Flag 25yr Less than 5yr Flag 100yr Less than 25yr  \
1  48.83055993680345

In [10]:
# 4. Merge Population + Flood Dataset

df_combined = pd.merge(
    df_pop,
    df_flood,
    on="Barangay",
    how="inner"
)

print("\nAfter 2024 population + flood merge:")
print("Rows:", len(df_combined))
print(df_combined.head())



After 2024 population + flood merge:
Rows: 897
     Barangay  Population 2024   Barangay Area Sqm  Flood Area 5yr Sqm  \
0  Barangay 1           3119.0    49547.4905221462   8264.845529826996   
1  Barangay 2           1666.0  37885.968608624346   5076.648009076795   
2  Barangay 3           1232.0    38940.5616643151   27962.08709355034   
3  Barangay 4           1717.0   40193.68735214498   9028.712129733307   
4  Barangay 5           1321.0  24741.847790367072  6958.3624583700985   

        Flood PCT 5yr Flood Area 25yr Sqm     Flood PCT 25yr  \
0  16.680654141571235  21069.565046191274  42.52398017367565   
1  13.399810524894825  16510.818939359135  43.58030042711015   
2   71.80709753135027  37710.275790402564   96.8406057300402   
3   22.46301029967951   26647.36186006734  66.29738054785679   
4  28.123859290247715  17729.880743596867  71.65948515171034   

  Flood Area 100yr Sqm    Flood PCT 100yr Flag 25yr Less than 5yr  \
0    24194.31705659861  48.83055993680345            

In [11]:
# 5. Convert Core Numeric Columns

core_numeric_cols = [
    "Population 2024",
    "Barangay Area Sqm",
    "Flood Area 5yr Sqm",
    "Flood PCT 5yr",
    "Flood Area 25yr Sqm",
    "Flood PCT 25yr",
    "Flood Area 100yr Sqm",
    "Flood PCT 100yr"
]

df_combined = safe_numeric(df_combined, core_numeric_cols)

In [12]:
# 6. Sort by Barangay Number

df_combined["Sort_Key"] = df_combined["Barangay"].apply(extract_number)
df_combined = (
    df_combined
    .sort_values(by=["Sort_Key", "Barangay"])
    .drop(columns=["Sort_Key"])
    .reset_index(drop=True)
)

print("\nSorted combined dataset sample:")
print(df_combined.head())


Sorted combined dataset sample:
     Barangay  Population 2024  Barangay Area Sqm  Flood Area 5yr Sqm  \
0  Barangay 1           3119.0       49547.490522         8264.845530   
1  Barangay 2           1666.0       37885.968609         5076.648009   
2  Barangay 3           1232.0       38940.561664        27962.087094   
3  Barangay 4           1717.0       40193.687352         9028.712130   
4  Barangay 5           1321.0       24741.847790         6958.362458   

   Flood PCT 5yr  Flood Area 25yr Sqm  Flood PCT 25yr  Flood Area 100yr Sqm  \
0      16.680654         21069.565046       42.523980          24194.317057   
1      13.399811         16510.818939       43.580300          27996.492076   
2      71.807098         37710.275790       96.840606          38350.775469   
3      22.463010         26647.361860       66.297381          37927.322907   
4      28.123859         17729.880744       71.659485          23759.844830   

   Flood PCT 100yr Flag 25yr Less than 5yr Flag 100yr

In [13]:
# 7. Load Updated Base/Admin Dataset

df_base = pd.read_csv(base_path)

# Normalize base column names
df_base.columns = (
    df_base.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
)

print("\nUpdated base dataset loaded.")
print("Base shape:", df_base.shape)
print("Base columns:")
print(df_base.columns.tolist())


Updated base dataset loaded.
Base shape: (899, 74)
Base columns:
['fid', 'objectid', 'gthcode', 'globalid', 'st_area(shape)', 'reg_name', 'prov_name', 'city_name', 'brgy_name', 'reg_code', 'prov_code', 'city_code', 'brgy_code', 'psgc_10d', 'geographic_level', 'old_names', 'class', 'income_class', 'urban_rural2015', 'bgypop2015', 'bgypop2020', 'bgyarea_sqkm', 'gida', 'gida_yr', 'st_perimeter(shape)', 'created_user', 'created_date', 'last_edited_user', 'last_edited_date', 'flood5_summary_fid', 'flood5_summary_count', 'flood5_summary_unique', 'flood5_summary_min', 'flood5_summary_max', 'flood5_summary_range', 'flood5_summary_sum', 'flood5_summary_mean', 'flood5_summary_median', 'flood5_summary_stddev', 'flood5_summary_minority', 'flood5_summary_majority', 'flood5_summary_q1', 'flood5_summary_q3', 'flood5_summary_iqr', 'flood25_summary_fid', 'flood25_summary_count', 'flood25_summary_unique', 'flood25_summary_min', 'flood25_summary_max', 'flood25_summary_range', 'flood25_summary_sum', 'flo

In [14]:
#  8. Detect Barangay Column in Base Dataset

possible_barangay_cols = [
    "brgy_name",
    "barangay",
    "barangay_name",
    "brgy",
    "barangayname"
]

base_barangay_col = None

for col in possible_barangay_cols:
    if col in df_base.columns:
        base_barangay_col = col
        break

if base_barangay_col is None:
    raise ValueError(
        "No barangay name column found in base dataset. "
        "Check df_base.columns.tolist()."
    )

print("\nBase barangay column detected:", base_barangay_col)


Base barangay column detected: brgy_name


In [15]:
#  9. Create Merge Keys

df_combined["barangay_clean"] = df_combined["Barangay"].apply(clean_barangay_name)
df_combined["barangay_key_strict"] = df_combined["Barangay"].apply(clean_barangay_name_strict)

df_base["barangay_clean"] = df_base[base_barangay_col].apply(clean_barangay_name)
df_base["barangay_key_strict"] = df_base[base_barangay_col].apply(clean_barangay_name_strict)

In [16]:
# 10. Select Useful Columns from Updated Base Dataset

base_cols_to_add = [
    "barangay_key_strict",
    "barangay_clean",

    # Official/admin metadata
    "psgc_10d",
    "reg_name",
    "reg_code",
    "prov_name",
    "prov_code",
    "city_name",
    "city_code",
    "brgy_code",
    "geographic_level",
    "class",
    "urban_rural2015",
    "gida",
    "gida_yr",

    # Demographic and area
    "bgypop2015",
    "bgypop2020",
    "bgyarea_sqkm",

    # Optional identifiers/reference
    "fid",
    "objectid",
    "gthcode",
    "globalid",
    "st_area(shape)",
    "st_perimeter(shape)"
]

optional_flood_summary_cols = [
    # 5-year summary
    "flood5_summary_count",
    "flood5_summary_unique",
    "flood5_summary_min",
    "flood5_summary_max",
    "flood5_summary_range",
    "flood5_summary_sum",
    "flood5_summary_mean",
    "flood5_summary_median",
    "flood5_summary_stddev",
    "flood5_summary_q1",
    "flood5_summary_q3",
    "flood5_summary_iqr",

    # 25-year summary
    "flood25_summary_count",
    "flood25_summary_unique",
    "flood25_summary_min",
    "flood25_summary_max",
    "flood25_summary_range",
    "flood25_summary_sum",
    "flood25_summary_mean",
    "flood25_summary_median",
    "flood25_summary_stddev",
    "flood25_summary_q1",
    "flood25_summary_q3",
    "flood25_summary_iqr",

    # 100-year summary
    "flood100_summary_count",
    "flood100_summary_unique",
    "flood100_summary_min",
    "flood100_summary_max",
    "flood100_summary_range",
    "flood100_summary_sum",
    "flood100_summary_mean",
    "flood100_summary_median",
    "flood100_summary_stddev",
    "flood100_summary_q1",
    "flood100_summary_q3",
    "flood100_summary_iqr"
]

base_cols_existing = [
    col for col in base_cols_to_add + optional_flood_summary_cols
    if col in df_base.columns
]

print("\nBase columns selected for merge:")
print(base_cols_existing)

df_base_selected = df_base[base_cols_existing].copy()

# Remove duplicate barangay records from base dataset
df_base_selected = df_base_selected.drop_duplicates(subset=["barangay_key_strict"])


Base columns selected for merge:
['barangay_key_strict', 'barangay_clean', 'psgc_10d', 'reg_name', 'reg_code', 'prov_name', 'prov_code', 'city_name', 'city_code', 'brgy_code', 'geographic_level', 'class', 'urban_rural2015', 'gida', 'gida_yr', 'bgypop2015', 'bgypop2020', 'bgyarea_sqkm', 'fid', 'objectid', 'gthcode', 'globalid', 'st_area(shape)', 'st_perimeter(shape)', 'flood5_summary_count', 'flood5_summary_unique', 'flood5_summary_min', 'flood5_summary_max', 'flood5_summary_range', 'flood5_summary_sum', 'flood5_summary_mean', 'flood5_summary_median', 'flood5_summary_stddev', 'flood5_summary_q1', 'flood5_summary_q3', 'flood5_summary_iqr', 'flood25_summary_count', 'flood25_summary_unique', 'flood25_summary_min', 'flood25_summary_max', 'flood25_summary_range', 'flood25_summary_sum', 'flood25_summary_mean', 'flood25_summary_median', 'flood25_summary_stddev', 'flood25_summary_q1', 'flood25_summary_q3', 'flood25_summary_iqr', 'flood100_summary_count', 'flood100_summary_unique', 'flood100_su

In [17]:
# 11. Merge Updated Base Features Into Main Dataset

df_combined = pd.merge(
    df_combined,
    df_base_selected,
    on="barangay_key_strict",
    how="left",
    suffixes=("", "_base")
)

# If barangay_clean_base exists after merge, keep one consistent value.
if "barangay_clean_base" in df_combined.columns:
    df_combined = df_combined.drop(columns=["barangay_clean_base"])

print("\nAfter adding updated base dataset features:")
print("Rows:", len(df_combined))
print("Columns:", len(df_combined.columns))

if "city_name" in df_combined.columns:
    print("Matched base records:", df_combined["city_name"].notna().sum())
    print("Unmatched base records:", df_combined["city_name"].isna().sum())

    unmatched = df_combined[df_combined["city_name"].isna()][["Barangay", "barangay_clean", "barangay_key_strict"]]
    print("\nUnmatched barangays preview:")
    print(unmatched.head(50))


After adding updated base dataset features:
Rows: 897
Columns: 73
Matched base records: 897
Unmatched base records: 0

Unmatched barangays preview:
Empty DataFrame
Columns: [Barangay, barangay_clean, barangay_key_strict]
Index: []


In [18]:
# 12. Convert Base Numeric Columns

base_numeric_cols = [
    "bgypop2015",
    "bgypop2020",
    "bgyarea_sqkm",

    "flood5_summary_count",
    "flood5_summary_unique",
    "flood5_summary_min",
    "flood5_summary_max",
    "flood5_summary_range",
    "flood5_summary_sum",
    "flood5_summary_mean",
    "flood5_summary_median",
    "flood5_summary_stddev",
    "flood5_summary_q1",
    "flood5_summary_q3",
    "flood5_summary_iqr",

    "flood25_summary_count",
    "flood25_summary_unique",
    "flood25_summary_min",
    "flood25_summary_max",
    "flood25_summary_range",
    "flood25_summary_sum",
    "flood25_summary_mean",
    "flood25_summary_median",
    "flood25_summary_stddev",
    "flood25_summary_q1",
    "flood25_summary_q3",
    "flood25_summary_iqr",

    "flood100_summary_count",
    "flood100_summary_unique",
    "flood100_summary_min",
    "flood100_summary_max",
    "flood100_summary_range",
    "flood100_summary_sum",
    "flood100_summary_mean",
    "flood100_summary_median",
    "flood100_summary_stddev",
    "flood100_summary_q1",
    "flood100_summary_q3",
    "flood100_summary_iqr"
]

df_combined = safe_numeric(df_combined, base_numeric_cols)

In [19]:
# ============================================================
# 13. Core Feature Engineering Using 2024 Population
# ============================================================

# Population density per square meter using 2024 population
df_combined["Population Density 2024 per Sqm"] = np.where(
    df_combined["Barangay Area Sqm"] > 0,
    df_combined["Population 2024"] / df_combined["Barangay Area Sqm"],
    np.nan
)

# Population density per hectare using 2024 population
df_combined["Population Density 2024 per Hectare"] = np.where(
    df_combined["Barangay Area Sqm"] > 0,
    df_combined["Population 2024"] / (df_combined["Barangay Area Sqm"] / 10000),
    np.nan
)

# Optional compatibility alias
# Use this if later code expects "Population Density per Hectare"
df_combined["Population Density per Hectare"] = df_combined["Population Density 2024 per Hectare"]

# Flood ratios
df_combined["Flood Ratio 5yr"] = df_combined["Flood PCT 5yr"] / 100
df_combined["Flood Ratio 25yr"] = df_combined["Flood PCT 25yr"] / 100
df_combined["Flood Ratio 100yr"] = df_combined["Flood PCT 100yr"] / 100

# Affected population using 2024 population
df_combined["Affected Population 2024 5yr"] = (
    df_combined["Population 2024"] * df_combined["Flood Ratio 5yr"]
).round(0).astype("Int64")

df_combined["Affected Population 2024 25yr"] = (
    df_combined["Population 2024"] * df_combined["Flood Ratio 25yr"]
).round(0).astype("Int64")

df_combined["Affected Population 2024 100yr"] = (
    df_combined["Population 2024"] * df_combined["Flood Ratio 100yr"]
).round(0).astype("Int64")

# Optional compatibility aliases
# Use these if later code expects the old names
df_combined["Affected Population 5yr"] = df_combined["Affected Population 2024 5yr"]
df_combined["Affected Population 25yr"] = df_combined["Affected Population 2024 25yr"]
df_combined["Affected Population 100yr"] = df_combined["Affected Population 2024 100yr"]

# Flood escalation percentages
df_combined["Flood Escalation 5to25"] = (
    df_combined["Flood PCT 25yr"] - df_combined["Flood PCT 5yr"]
)

df_combined["Flood Escalation 25to100"] = (
    df_combined["Flood PCT 100yr"] - df_combined["Flood PCT 25yr"]
)

df_combined["Flood Escalation 5to100"] = (
    df_combined["Flood PCT 100yr"] - df_combined["Flood PCT 5yr"]
)

# Flood area severity increases
df_combined["Flood Severity Increase 5to25 Sqm"] = (
    df_combined["Flood Area 25yr Sqm"] - df_combined["Flood Area 5yr Sqm"]
)

df_combined["Flood Severity Increase 25to100 Sqm"] = (
    df_combined["Flood Area 100yr Sqm"] - df_combined["Flood Area 25yr Sqm"]
)

df_combined["Flood Severity Increase 5to100 Sqm"] = (
    df_combined["Flood Area 100yr Sqm"] - df_combined["Flood Area 5yr Sqm"]
)

# Affected population increases using 2024 population
df_combined["Affected Population Increase 2024 5to25"] = (
    df_combined["Affected Population 2024 25yr"] -
    df_combined["Affected Population 2024 5yr"]
)

df_combined["Affected Population Increase 2024 25to100"] = (
    df_combined["Affected Population 2024 100yr"] -
    df_combined["Affected Population 2024 25yr"]
)

df_combined["Affected Population Increase 2024 5to100"] = (
    df_combined["Affected Population 2024 100yr"] -
    df_combined["Affected Population 2024 5yr"]
)

# Optional compatibility aliases
df_combined["Affected Population Increase 5to25"] = df_combined["Affected Population Increase 2024 5to25"]
df_combined["Affected Population Increase 25to100"] = df_combined["Affected Population Increase 2024 25to100"]
df_combined["Affected Population Increase 5to100"] = df_combined["Affected Population Increase 2024 5to100"]


In [20]:
# ============================================================
# 14. Demographic Features: 2020 to 2024
# ============================================================

if "bgypop2020" in df_combined.columns and "Population 2024" in df_combined.columns:

    # Population growth from 2020 to 2024
    df_combined["Population Growth 2020to2024"] = (
        df_combined["Population 2024"] - df_combined["bgypop2020"]
    )

    df_combined["Population Growth Rate 2020to2024"] = np.where(
        df_combined["bgypop2020"] > 0,
        (
            (df_combined["Population 2024"] - df_combined["bgypop2020"])
            / df_combined["bgypop2020"]
        ) * 100,
        np.nan
    )

    # Density based on 2020 official base population
    df_combined["Density 2020 per Hectare"] = np.where(
        df_combined["Barangay Area Sqm"] > 0,
        df_combined["bgypop2020"] / (df_combined["Barangay Area Sqm"] / 10000),
        np.nan
    )

    # Density based on 2024 population
    df_combined["Density 2024 per Hectare"] = np.where(
        df_combined["Barangay Area Sqm"] > 0,
        df_combined["Population 2024"] / (df_combined["Barangay Area Sqm"] / 10000),
        np.nan
    )

    df_combined["Density Growth 2020to2024 per Hectare"] = (
        df_combined["Density 2024 per Hectare"] -
        df_combined["Density 2020 per Hectare"]
    )

    # Affected population growth using 25-year flood percentage
    df_combined["Affected Pop 2020 25yr"] = (
        df_combined["bgypop2020"] * df_combined["Flood PCT 25yr"] / 100
    ).round(0).astype("Int64")

    df_combined["Affected Pop 2024 25yr"] = (
        df_combined["Population 2024"] * df_combined["Flood PCT 25yr"] / 100
    ).round(0).astype("Int64")

    df_combined["Affected Pop Growth 2020to2024 25yr"] = (
        df_combined["Affected Pop 2024 25yr"] -
        df_combined["Affected Pop 2020 25yr"]
    )

    print("2020–2024 demographic growth features created.")

else:
    print("Cannot compute 2020–2024 growth. Missing bgypop2020 or Population 2024.")


2020–2024 demographic growth features created.


In [21]:
# 15. Area Validation Features

if "bgyarea_sqkm" in df_combined.columns:

    df_combined["Base Area Sqm"] = df_combined["bgyarea_sqkm"] * 1_000_000

    df_combined["Area Difference Sqm"] = (
        df_combined["Barangay Area Sqm"] - df_combined["Base Area Sqm"]
    )

    df_combined["Area Difference Pct"] = np.where(
        df_combined["Base Area Sqm"] > 0,
        (df_combined["Area Difference Sqm"] / df_combined["Base Area Sqm"]) * 100,
        np.nan
    )

In [22]:
# ============================================================
# 16. District-Level Features Using 2024 Density
# ============================================================

if "city_name" in df_combined.columns:

    df_combined["District Avg Flood PCT 5yr"] = (
        df_combined.groupby("city_name")["Flood PCT 5yr"].transform("mean")
    )

    df_combined["District Avg Flood PCT 25yr"] = (
        df_combined.groupby("city_name")["Flood PCT 25yr"].transform("mean")
    )

    df_combined["District Avg Population Density 2024 per Hectare"] = (
        df_combined.groupby("city_name")["Population Density 2024 per Hectare"].transform("mean")
    )

    # Optional compatibility alias
    df_combined["District Avg Population Density per Hectare"] = (
        df_combined["District Avg Population Density 2024 per Hectare"]
    )

    df_combined["Relative Flood Risk 25yr vs District"] = (
        df_combined["Flood PCT 25yr"] -
        df_combined["District Avg Flood PCT 25yr"]
    )

    df_combined["Relative Pop Density 2024 vs District"] = (
        df_combined["Population Density 2024 per Hectare"] -
        df_combined["District Avg Population Density 2024 per Hectare"]
    )

    # Optional compatibility alias
    df_combined["Relative Pop Density vs District"] = (
        df_combined["Relative Pop Density 2024 vs District"]
    )


In [23]:
# 17. Optional Flood Summary Derived Features

if "flood5_summary_mean" in df_combined.columns and "flood25_summary_mean" in df_combined.columns:
    df_combined["Flood Summary Mean Increase 5to25"] = (
        df_combined["flood25_summary_mean"] - df_combined["flood5_summary_mean"]
    )

if "flood5_summary_stddev" in df_combined.columns and "flood25_summary_stddev" in df_combined.columns:
    df_combined["Flood Summary StdDev Increase 5to25"] = (
        df_combined["flood25_summary_stddev"] - df_combined["flood5_summary_stddev"]
    )

if "flood5_summary_iqr" in df_combined.columns and "flood25_summary_iqr" in df_combined.columns:
    df_combined["Flood Summary IQR Increase 5to25"] = (
        df_combined["flood25_summary_iqr"] - df_combined["flood5_summary_iqr"]
    )

if "flood100_summary_mean" in df_combined.columns and "flood25_summary_mean" in df_combined.columns:
    df_combined["Flood Summary Mean Increase 25to100"] = (
        df_combined["flood100_summary_mean"] - df_combined["flood25_summary_mean"]
    )

In [24]:
# 18. 100-Year Validation Flags

df_combined["Flag 25yr Less than 5yr"] = (
    df_combined["Flood PCT 25yr"] < df_combined["Flood PCT 5yr"]
)

df_combined["Flag 100yr Less than 25yr"] = (
    df_combined["Flood PCT 100yr"] < df_combined["Flood PCT 25yr"]
)

df_combined["Flag 100yr Less than 5yr"] = (
    df_combined["Flood PCT 100yr"] < df_combined["Flood PCT 5yr"]
)

df_combined["100yr Data Quality Note"] = np.where(
    df_combined["Flag 100yr Less than 25yr"] | df_combined["Flag 100yr Less than 5yr"],
    "Review 100yr value: lower than expected return-period progression",
    "OK"
)

In [25]:
# 19. Hazard Indices

# Primary index excludes 100-year because of known inconsistencies.
df_combined["Primary Hazard Index 5yr25yr"] = (
    0.45 * df_combined["Flood PCT 5yr"] +
    0.55 * df_combined["Flood PCT 25yr"]
)

# Supplementary only; do not use as the main target unless 100-year data is validated.
df_combined["Supplementary Hazard Index 5yr25yr100yr"] = (
    0.25 * df_combined["Flood PCT 5yr"] +
    0.35 * df_combined["Flood PCT 25yr"] +
    0.40 * df_combined["Flood PCT 100yr"]
)

In [26]:
# ============================================================
# 20. Basic Data Quality Summary
# ============================================================

print("\n================ DATA QUALITY SUMMARY ================")
print("Final dataset shape:", df_combined.shape)

print("\n100-year lower than 25-year count:")
print(df_combined["Flag 100yr Less than 25yr"].sum())

print("\n100-year lower than 5-year count:")
print(df_combined["Flag 100yr Less than 5yr"].sum())

if "city_name" in df_combined.columns:
    print("\nDistrict counts:")
    print(df_combined["city_name"].value_counts(dropna=False))

if "Area Difference Pct" in df_combined.columns:
    print("\nArea difference percentage summary:")
    print(df_combined["Area Difference Pct"].describe())

if "Population Growth Rate 2020to2024" in df_combined.columns:
    print("\nPopulation growth rate 2020–2024 summary:")
    print(df_combined["Population Growth Rate 2020to2024"].describe())

if "Population 2024" in df_combined.columns:
    print("\nPopulation 2024 summary:")
    print(df_combined["Population 2024"].describe())



================ DATA QUALITY SUMMARY ================
Final dataset shape: (897, 122)

100-year lower than 25-year count:
590

100-year lower than 5-year count:
182

District counts:
city_name
City of Manila - Tondo I / II    259
City of Manila - Sampaloc        243
City of Manila - Santa Ana        99
City of Manila - Santa Cruz       82
City of Manila - Malate           57
City of Manila - Paco             43
City of Manila - Pandacan         38
City of Manila - Quiapo           16
City of Manila - San Nicolas      15
City of Manila - Ermita           13
City of Manila - San Miguel       12
City of Manila - Binondo          10
City of Manila - Intramuros        5
City of Manila - Port Area         5
Name: count, dtype: int64

Area difference percentage summary:
count    897.000000
mean       0.035516
std        0.015302
min       -0.301358
25%        0.034681
50%        0.035946
75%        0.037516
max        0.146044
Name: Area Difference Pct, dtype: float64

Population growth rat

In [27]:
# 20B. Create Clean ML-Ready Dataset by Dropping Redundant Features

# Columns to keep only for identification/reference
id_reference_cols = [
    "Barangay",
    "Barangay No",
    "psgc_10d",
    "city_name",
    "brgy_code"
]

# Recommended ML predictor features
ml_feature_cols = [
    # Hazard indicators using validated 5yr and 25yr layers
    "Flood PCT 5yr",
    "Flood PCT 25yr",
    "Flood Escalation 5to25",
    "Flood Severity Increase 5to25 Sqm",

    # Exposure indicators
    "Population",
    "Affected Population 5yr",
    "Affected Population 25yr",
    "Affected Population Increase 5to25",

    # Vulnerability indicators
    "Population Density per Hectare",
    "Population Growth Rate 2015to2020",

    # Optional demographic/context indicators
    "Density Growth 2015to2020 per Hectare",
    "Affected Pop Growth 25yr",

    # District/context features
    "District Avg Flood PCT 5yr",
    "District Avg Flood PCT 25yr",
    "District Avg Population Density per Hectare",
    "Relative Flood Risk 25yr vs District",
    "Relative Pop Density vs District",

    # Area validation/context, optional but useful for checking
    "Area Difference Pct"
]

# Add elevation feature here once available
# Example:
# ml_feature_cols.append("Mean Elevation")
# ml_feature_cols.append("Elevation Susceptibility Score")

# Keep only columns that exist in the dataframe
id_reference_existing = [col for col in id_reference_cols if col in df_combined.columns]
ml_feature_existing = [col for col in ml_feature_cols if col in df_combined.columns]

# Create ML-ready dataframe
df_ml_ready = df_combined[id_reference_existing + ml_feature_existing].copy()

print("\n================ ML-READY DATASET SUMMARY ================")
print("ML-ready shape:", df_ml_ready.shape)
print("Reference columns kept:")
print(id_reference_existing)
print("\nML feature columns kept:")
print(ml_feature_existing)

# Optional missing value check
print("\nMissing values in ML-ready dataset:")
print(df_ml_ready.isna().sum().sort_values(ascending=False).head(20))


================ ML-READY DATASET SUMMARY ================
ML-ready shape: (897, 19)
Reference columns kept:
['Barangay', 'Barangay No', 'psgc_10d', 'city_name', 'brgy_code']

ML feature columns kept:
['Flood PCT 5yr', 'Flood PCT 25yr', 'Flood Escalation 5to25', 'Flood Severity Increase 5to25 Sqm', 'Affected Population 5yr', 'Affected Population 25yr', 'Affected Population Increase 5to25', 'Population Density per Hectare', 'District Avg Flood PCT 5yr', 'District Avg Flood PCT 25yr', 'District Avg Population Density per Hectare', 'Relative Flood Risk 25yr vs District', 'Relative Pop Density vs District', 'Area Difference Pct']

Missing values in ML-ready dataset:
Barangay                                       0
Barangay No                                    0
psgc_10d                                       0
city_name                                      0
brgy_code                                      0
Flood PCT 5yr                                  0
Flood PCT 25yr                    

In [28]:
# 21. Export Final Dataset

# Full enriched dataset: keep everything for documentation/dashboard/appendix
full_output_csv = "/content/manila-city-flood-population_enriched_FULL.csv"
full_output_excel = "/content/manila-city-flood-population_enriched_FULL.xlsx"

# ML-ready dataset: selected usable features only
ml_output_csv = "/content/manila_ml_ready_dataset.csv"
ml_output_excel = "/content/manila_ml_ready_dataset.xlsx"

df_combined.to_csv(full_output_csv, index=False)
df_combined.to_excel(full_output_excel, index=False)

df_ml_ready.to_csv(ml_output_csv, index=False)
df_ml_ready.to_excel(ml_output_excel, index=False)

print("\n================ EXPORT COMPLETE ================")
print("Full CSV exported to:", full_output_csv)
print("Full Excel exported to:", full_output_excel)
print("ML-ready CSV exported to:", ml_output_csv)
print("ML-ready Excel exported to:", ml_output_excel)

print("\nFull dataset shape:", df_combined.shape)
print("ML-ready dataset shape:", df_ml_ready.shape)

display(df_ml_ready.head())


================ EXPORT COMPLETE ================
Full CSV exported to: /content/manila-city-flood-population_enriched_FULL.csv
Full Excel exported to: /content/manila-city-flood-population_enriched_FULL.xlsx
ML-ready CSV exported to: /content/manila_ml_ready_dataset.csv
ML-ready Excel exported to: /content/manila_ml_ready_dataset.xlsx

Full dataset shape: (897, 122)
ML-ready dataset shape: (897, 19)


,Barangay,Barangay No,psgc_10d,city_name,brgy_code,Flood PCT 5yr,Flood PCT 25yr,Flood Escalation 5to25,Flood Severity Increase 5to25 Sqm,Affected Population 5yr,Affected Population 25yr,Affected Population Increase 5to25,Population Density per Hectare,District Avg Flood PCT 5yr,District Avg Flood PCT 25yr,District Avg Population Density per Hectare,Relative Flood Risk 25yr vs District,Relative Pop Density vs District,Area Difference Pct
0,Barangay 1,1,1.380601e+09,City of Manila - Tondo I / II,133901001,16.680654,42.523980,25.843326,12804.719516,520,1326,806,629.497068,53.293173,74.009798,850.366607,-31.485818,-220.869539,0.038666
1,Barangay 2,2,1.380601e+09,City of Manila - Tondo I / II,133901002,13.399811,43.580300,30.180490,11434.170930,223,726,503,439.740638,53.293173,74.009798,850.366607,-30.429498,-410.625969,0.038627
2,Barangay 3,3,1.380601e+09,City of Manila - Tondo I / II,133901003,71.807098,96.840606,25.033508,9748.188697,885,1193,308,316.379617,53.293173,74.009798,850.366607,22.830808,-533.986990,0.038514
3,Barangay 4,4,1.380601e+09,City of Manila - Tondo I / II,133901004,22.463010,66.297381,43.834370,17618.649730,386,1138,752,427.181509,53.293173,74.009798,850.366607,-7.712417,-423.185098,0.038621
4,Barangay 5,5,1.380601e+09,City of Manila - Tondo I / II,133901005,28.123859,71.659485,43.535626,10771.518285,372,947,575,533.913235,53.293173,74.009798,850.366607,-2.350313,-316.453372,0.038685
